# Spark + Iceberg — основное

Каталог `hive` уже настроен в `spark-defaults.conf`: тип hive, метаданные в Hive Metastore,
данные в MinIO (`s3a://warehouse`). Ничего подключать руками не нужно.

Что показано: DDL, вставка, партиционирование, эволюция схемы, снапшоты, time travel, MERGE и метаданные Iceberg.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("quickstart").getOrCreate()
spark.sql("CREATE DATABASE IF NOT EXISTS hive.demo")
print("Spark", spark.version, "| UI: http://localhost:4040")

## Таблица с партиционированием по дате

In [ ]:
spark.sql("DROP TABLE IF EXISTS hive.demo.events")
spark.sql('''
  CREATE TABLE hive.demo.events (
    id      BIGINT,
    ts      TIMESTAMP,
    kind    STRING,
    payload STRING
  ) USING iceberg
  PARTITIONED BY (days(ts))      -- скрытое партиционирование Iceberg: по дню из ts
''')
spark.sql("DESCRIBE EXTENDED hive.demo.events").show(50, truncate=False)

## Вставка данных

In [ ]:
spark.sql('''
  INSERT INTO hive.demo.events VALUES
    (1, TIMESTAMP '2025-01-01 10:00:00', 'click', 'a'),
    (2, TIMESTAMP '2025-01-01 11:00:00', 'view',  'b'),
    (3, TIMESTAMP '2025-01-02 09:30:00', 'click', 'c')
''')
spark.table("hive.demo.events").show(truncate=False)

## Эволюция схемы — добавить колонку без перезаписи данных

In [ ]:
spark.sql("ALTER TABLE hive.demo.events ADD COLUMN source STRING")
spark.sql("INSERT INTO hive.demo.events VALUES (4, TIMESTAMP '2025-01-02 12:00:00', 'view', 'd', 'web')")
spark.table("hive.demo.events").show(truncate=False)

## MERGE (upsert) — то, ради чего часто и берут Iceberg

In [ ]:
spark.sql("CREATE OR REPLACE TEMP VIEW updates AS SELECT * FROM VALUES (1, 'click_v2'), (5, 'scroll') AS t(id, kind)")
spark.sql('''
  MERGE INTO hive.demo.events e USING updates u ON e.id = u.id
  WHEN MATCHED THEN UPDATE SET e.kind = u.kind
  WHEN NOT MATCHED THEN INSERT (id, kind) VALUES (u.id, u.kind)
''')
spark.sql("SELECT id, kind FROM hive.demo.events ORDER BY id").show()

## Снапшоты и история — каждое изменение это новый снапшот

In [ ]:
spark.sql("SELECT snapshot_id, committed_at, operation FROM hive.demo.events.snapshots ORDER BY committed_at").show(truncate=False)

## Time travel — прочитать таблицу, какой она была на первом снапшоте

In [ ]:
first = spark.sql("SELECT snapshot_id sid FROM hive.demo.events.snapshots ORDER BY committed_at LIMIT 1").collect()[0]["sid"]
print("первый снапшот:", first)
spark.read.option("snapshot-id", first).table("hive.demo.events").orderBy("id").show()

## Служебные метаданные: файлы и партиции

In [ ]:
spark.sql("SELECT file_path, record_count, file_size_in_bytes FROM hive.demo.events.files").show(truncate=False)
spark.sql("SELECT partition, record_count FROM hive.demo.events.partitions").show(truncate=False)

Данные лежат в MinIO — видно в консоли http://localhost:9001 под `warehouse/demo.db/events`.